# **CODE ASSISTANT USING LANGCHAIN :**

In [1]:
pip install langchain-community

Note: you may need to restart the kernel to use updated packages.


## Import libraries :

In [2]:
import os
import langchain
from langchain.schema import HumanMessage
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains import LLMChain,ConversationChain
from langchain.memory import ConversationBufferMemory

In [3]:
import ast 
import traceback

## Groq api key :

In [4]:
os.environ["GROQ_API_KEY"]="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"

## Prompt Template :

In [5]:
template = """
You are a helpful coding assistant. Answer the user's queries with Python code examples,
explanations, and best practices. 

User Query: {query}
"""


In [6]:
prompt=ChatPromptTemplate.from_template(template)

## Initialize LLM :

In [7]:
!pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langchain-groq]m [langchain-groq]


In [8]:
from langchain_groq import ChatGroq

In [9]:
llm=ChatGroq(
    model="gemma2-9b-it",
    temperature=0.2,
    api_key=os.environ["GROQ_API_KEY"]
)

In [10]:
response=llm.invoke("Write a python code for greatest numbers between two numbers")
print(response.content)

```python
def find_greatest(num1, num2):
  """
  This function finds the greatest number between two given numbers.

  Args:
    num1: The first number.
    num2: The second number.

  Returns:
    The greatest number between num1 and num2.
  """
  if num1 > num2:
    return num1
  else:
    return num2

# Get input from the user
num1 = float(input("Enter the first number: "))
num2 = float(input("Enter the second number: "))

# Find the greatest number
greatest_number = find_greatest(num1, num2)

# Print the result
print("The greatest number is:", greatest_number)
```

**Explanation:**

1. **Define a function:**
   - `find_greatest(num1, num2)` takes two numbers as input.
   - It uses an `if` statement to compare `num1` and `num2`.
   - If `num1` is greater, it returns `num1`.
   - Otherwise, it returns `num2`.

2. **Get user input:**
   - `input()` prompts the user to enter two numbers.
   - `float()` converts the input strings to floating-point numbers to handle potential decimal val

## Setup memory for Chat history :

In [11]:
memory=ConversationBufferMemory(memory_key="chat_history",return_messages=True)

/tmp/ipykernel_33665/463180671.py:1: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory=ConversationBufferMemory(memory_key="chat_history",return_messages=True)


## LLM Chain :

In [12]:
chain=LLMChain(
    llm=llm,
    prompt=prompt,
    memory=memory
)

/tmp/ipykernel_33665/763293122.py:1: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain=LLMChain(


In [13]:
response=chain.run("Write a python code for greatest numbers between two numbers")
print(response)

/tmp/ipykernel_33665/197252661.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response=chain.run("Write a python code for greatest numbers between two numbers")


```python
def find_greatest(num1, num2):
  """
  This function finds the greatest number between two given numbers.

  Args:
    num1: The first number.
    num2: The second number.

  Returns:
    The greatest number between num1 and num2.
  """
  if num1 > num2:
    return num1
  else:
    return num2

# Get input from the user
num1 = float(input("Enter the first number: "))
num2 = float(input("Enter the second number: "))

# Find the greatest number
greatest_number = find_greatest(num1, num2)

# Print the result
print("The greatest number is:", greatest_number)
```

**Explanation:**

1. **Function Definition:**
   - We define a function called `find_greatest` that takes two arguments, `num1` and `num2`.
   - Inside the function, we use an `if` statement to compare the two numbers.
   - If `num1` is greater than `num2`, we return `num1`. Otherwise, we return `num2`.

2. **User Input:**
   - We use `input()` to get two numbers from the user and convert them to floating-point numbers u

## Python Code Execution Helper :

In [14]:
import io
import sys
import ast
import traceback
import re

def execute_code(code_str, input_values=None):
    """
    Execute Python code safely with input() replacement.
    """
    input_values = input_values or []
    input_counter = 0

    def replace_input(match):
        nonlocal input_counter
        if input_counter < len(input_values):
            val = input_values[input_counter]
            input_counter += 1
            # Strings wrapped in quotes
            if isinstance(val, str):
                return f'"{val}"'
            else:
                return str(val)
        else:
            # Smart default based on context:
            # If 'int(' in the line, return 1; if 'float(', return 1.0; else 'done'
            line = match.string
            if 'int(' in line:
                return '1'
            elif 'float(' in line:
                return '1.0'
            else:
                return '"done"'

    pattern = r'input\s*\([^\)]*\)'
    code_str = re.sub(pattern, replace_input, code_str)

    old_stdout = sys.stdout
    try:
        sys.stdout = io.StringIO()
        tree = ast.parse(code_str)
        compiled = compile(tree, filename="<ast>", mode="exec")
        exec_locals = {}
        exec(compiled, {}, exec_locals)
        output = sys.stdout.getvalue()
        sys.stdout = old_stdout
        return output if output else str(exec_locals)
    except Exception:
        sys.stdout = old_stdout
        return traceback.format_exc()


## Main Assistant Function :

In [15]:
def code_assistant(user_input, chain, input_values=None):
    response = chain.run(user_input)

    if "```python" in response:
        code_block = response.split("```python")[1].split("```")[0]
        output = execute_code(code_block, input_values=input_values)
        return f"{response}\n\n--- Output ---\n{output}"

    return response


## Gradio Interface :

In [16]:
import gradio as gr

/home/uma/anaconda3/envs/tfod/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
# Gradio wrapper
def code_assistant_gradio(user_input):
    return code_assistant(user_input, chain)  # Make sure 'chain' is defined

# Gradio UI
interface = gr.Interface(
    fn=code_assistant_gradio,
    inputs=gr.Textbox(lines=5, placeholder="Ask me to write Python code or explain Python concepts..."),
    outputs=gr.Textbox(lines=15),
    title="Python Code Assistant",
    description="A Python coding assistant built using LangChain and Groq/OpenAI API. It can write, debug, and explain Python code.",
    allow_flagging="never"
)

interface.launch()


Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


In [18]:
def get_chain():
    return chain